<a href="https://colab.research.google.com/github/WallofWonder/Hugging-Face-RL-Learning/blob/main/notebooks/unit5/unit5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 单元 5: ML-Agents 介绍

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/thumbnail.png" alt="Thumbnail"/>

在本笔记本中，您将学习 ML-Agents 并训练两个智能体（Agent）。

- 第一个智能体将学习**向生成的靶子发射雪球**。
- 第二个智能体需要按下按钮来生成金字塔，然后导航到金字塔，将其撞倒，**并移动到顶部的金砖**。为了实现这一目标，它需要探索环境，我们将使用一种称为“好奇心”（Curiosity）的技术。

之后，您将能够**直接在浏览器中观看您的智能体进行游戏**。

关于认证过程的更多信息，请查看此部分 👉 https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process

⬇️ 这是您在**本单元结束时将实现的效果示例**。 ⬇️

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/pyramids.gif" alt="Pyramids"/>

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/snowballtarget.gif" alt="SnowballTarget"/>

### 🎮 环境：

- [Pyramids](https://github.com/Unity-Technologies/ml-agents/blob/main/docs/Learning-Environment-Examples.md#pyramids)
- SnowballTarget

### 📚 强化学习库：

- [ML-Agents](https://github.com/Unity-Technologies/ml-agents)

我们一直在努力改进我们的教程，因此**如果您在此笔记本中发现一些问题**，请[在 GitHub 仓库上提交 Issue](https://github.com/huggingface/deep-rl-class/issues)。

## 本笔记本的目标 🏆

在学习完本笔记本后，您将：

- 理解环境库 **ML-Agents** 的工作原理。
- 能够**在 Unity 环境中训练智能体**。

## 本笔记本来自《深度强化学习课程》
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/deep-rl-course-illustration.jpg" alt="Deep RL Course illustration"/>

在这门免费课程中，您将：

- 📖 从**理论和实践**上学习深度强化学习。
- 🧑‍💻 学习**使用著名的深度强化学习库**，如 Stable Baselines3、RL Baselines3 Zoo、CleanRL 和 Sample Factory 2.0。
- 🤖 在**独特的游戏环境中**训练智能体。

更多内容请查看 📚 教学大纲 👉 https://huggingface.co/deep-rl-course/communication/publishing-schedule

不要忘记**<a href="http://eepurl.com/ic5ZUD">注册课程</a>**（我们会收集您的电子邮箱，以便在发布每个单元时向您发送链接，并向您提供有关挑战和更新的信息）。

保持联系的最佳方式是加入我们的 Discord 服务器，与社区及我们进行交流 👉🏻 https://discord.gg/ydHrjt3WP5

## 前提条件 🏗️
在深入研究笔记本之前，您需要：

🔲 📚 **通过阅读第 5 单元来学习 [什么是 ML-Agents 以及它如何工作](https://huggingface.co/deep-rl-course/unit5/introduction)** 🤗

# 让我们开始训练我们的智能体 🚀

**要通过此动手实验来获得认证，您只需将训练好的模型推送到 Hugging Face Hub**。要通过此项认证，并没有硬性的分数指标要求。但如果您想获得出色的效果，可以尝试达到以下指标：

- 对于 `Pyramids`（金字塔）：平均奖励 = 1.75
- 对于 `SnowballTarget`（雪球打靶）：平均奖励 = 15，或在单个回合中击中 30 个目标。

## 设置 GPU 💪
- 为了**加速智能体的训练，我们将使用 GPU**。为此，请转到菜单 `Runtime > Change Runtime type`（代码执行程序 > 更改运行时类型）

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step1.jpg" alt="GPU Step 1">

- 选择 `Hardware Accelerator > GPU`（硬件加速器 > GPU）

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step2.jpg" alt="GPU Step 2">

## 克隆代码库 🔽

- 我们需要克隆包含 **ML-Agents** 的代码仓库。

In [ ]:
%%capture
# 克隆仓库（大约需要3分钟）
!git clone --depth 1 https://github.com/Unity-Technologies/ml-agents

## 设置虚拟环境 🔽
- 为了让 **ML-Agents** 在 Colab 中成功运行，Colab 的 Python 版本必须满足该库的 Python 版本要求。

- 我们可以在 `setup.py` 文件中的 `python_requires` 参数下查看受支持的 Python 版本。这些文件是设置和使用 **ML-Agents** 库所必需的，可以在以下位置找到：
  - `/content/ml-agents/ml-agents/setup.py`
  - `/content/ml-agents/ml-agents-envs/setup.py`

- Colab 当前的 Python 版本（可以使用 `!python --version` 查看）可能与库的 `python_requires` 参数不匹配，因此安装可能会静默失败，并导致稍后执行相同命令时出现以下错误：
  - `/bin/bash: line 1: mlagents-learn: command not found`
  - `/bin/bash: line 1: mlagents-push-to-hf: command not found`

- 为了解决这个问题，我们将创建一个与 **ML-Agents** 库兼容的 Python 版本的虚拟环境。

`注意：` *为了保证未来的兼容性，如果 Colab 的 Python 版本不兼容，请务必查看安装文件中的 `python_requires` 参数，并在下方的脚本中将您的虚拟环境设置为支持的最高 Python 版本*

In [ ]:
# Colab 当前的 Python 版本（与 ML-Agents 不兼容）
!python --version

In [ ]:
# 安装 virtualenv 并创建虚拟环境
!pip install virtualenv
!virtualenv myenv

# 下载并安装 Miniconda
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!chmod +x Miniconda3-latest-Linux-x86_64.sh
!./Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local

# 激活 Miniconda 并安装 Python 3.10.12 版本
!source /usr/local/bin/activate
!conda install -q -y --prefix /usr/local python=3.10.12 ujson  # 在此处指定版本

# 为 Python 和 conda 路径设置环境变量
!export PYTHONPATH=/usr/local/lib/python3.10/site-packages/
!export CONDA_PREFIX=/usr/local/envs/myenv

In [ ]:
# 新虚拟环境中的 Python 版本（与 ML-Agents 兼容）
!python --version

## 安装依赖项 🔽

In [ ]:
%%capture
# 进入仓库并安装软件包（大约需要3分钟）
%cd ml-agents
!pip3 install -e ./ml-agents-envs
!pip3 install -e ./ml-agents

## SnowballTarget ⛄ (雪球打靶)

如果您需要复习这些环境的工作原理，请查看本节 👉
https://huggingface.co/deep-rl-course/unit5/snowball-target

### 下载环境压缩包并移动到 `./training-envs-executables/linux/` 目录下
- 我们的环境可执行文件打包在 zip 文件中。
- 我们需要下载它并将其放置在 `./training-envs-executables/linux/` 目录中。
- 我们使用 Linux 可执行文件是因为我们使用的是 Colab，而 Colab 机器的操作系统是 Ubuntu (Linux)。

In [ ]:
# 在这里，我们创建 training-envs-executables 和 linux 目录
!mkdir ./training-envs-executables
!mkdir ./training-envs-executables/linux

我们使用 `wget` 从 https://github.com/huggingface/Snowball-Target 下载 SnowballTarget.zip 文件。

In [ ]:
!wget "https://github.com/huggingface/Snowball-Target/raw/main/SnowballTarget.zip" -O ./training-envs-executables/linux/SnowballTarget.zip

我们解压 executable.zip 文件。

In [ ]:
%%capture
!unzip -d ./training-envs-executables/linux/ ./training-envs-executables/linux/SnowballTarget.zip

确保您的文件是可执行且可访问的。

In [ ]:
!chmod -R 755 ./training-envs-executables/linux/SnowballTarget

### 定义 SnowballTarget 配置文件
- 在 ML-Agents 中，您需要在 **config.yaml 文件中定义训练超参数。**

这里有多个超参数。要更好地了解它们，您应该查看[官方文档](https://github.com/Unity-Technologies/ml-agents/blob/release_20_docs/docs/Training-Configuration-File.md)中的每个解释。

所以您需要在 `./content/ml-agents/config/ppo/` 目录下创建一个 `SnowballTarget.yaml` 配置文件。

我们将在这里为您提供该配置的第一个版本（复制并粘贴到您的 `SnowballTarget.yaml` 文件中），**但您应该尝试对其进行修改**。

```
behaviors:
  SnowballTarget:
    trainer_type: ppo
    summary_freq: 10000
    keep_checkpoints: 10
    checkpoint_interval: 50000
    max_steps: 200000
    time_horizon: 64
    threaded: false
    hyperparameters:
      learning_rate: 0.0003
      learning_rate_schedule: linear
      batch_size: 128
      buffer_size: 2048
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
    network_settings:
      normalize: false
      hidden_units: 256
      num_layers: 2
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.99
        strength: 1.0
```

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/snowballfight_config1.png" alt="Config SnowballTarget"/>
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/snowballfight_config2.png" alt="Config SnowballTarget"/>

作为实验，您还应该尝试修改其他一些超参数。Unity 提供了非常[优秀的文档，在这里详细解释了每一个超参数](https://github.com/Unity-Technologies/ml-agents/blob/main/docs/Training-Configuration-File.md)。

现在您已经创建了配置文件并了解了大多数超参数的作用，我们准备开始训练我们的智能体 🔥。

### 训练智能体

要训练我们的智能体，我们只需要**启动 mlagents-learn 并选择包含环境的可执行文件。**

我们定义了四个参数：

1. `mlagents-learn <config>`：超参数配置文件的路径。
2. `--env`：环境可执行文件的路径。
3. `--run_id`：您想要给训练运行指定的 ID 名称。
4. `--no-graphics`：在训练期间不启动图形界面可视化。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/mlagentslearn.png" alt="MlAgents learn"/>

训练模型并使用 `--resume` 标志以便在发生中断时继续训练。

> 在您使用 `--resume` 时，它第一次可能会失败，请尝试再次运行该单元以绕过错误。

训练将持续 10 到 35 分钟，具体取决于您的配置，去喝杯 ☕️ 吧，这是你应得的 🤗。

In [ ]:
!mlagents-learn ./config/ppo/SnowballTarget.yaml --env=./training-envs-executables/linux/SnowballTarget/SnowballTarget --run-id="SnowballTarget1" --no-graphics

### 将智能体推送到 🤗 Hub

- 现在我们已经训练好了智能体，我们**准备好将它推送到 Hub，以便能够在浏览器中可视化观看它玩游戏🔥。**

为了能够与社区分享您的模型，还有另外三个步骤需要遵循：

1️⃣ （如果尚未完成）在 HF 上创建一个账户 ➡ https://huggingface.co/join

2️⃣ 登录，然后您需要存储来自 Hugging Face 网站的身份验证 Token。
- 创建一个具有 **write (写入) 权限**的新 Token (https://huggingface.co/settings/tokens)

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/create-token.jpg" alt="Create HF Token">

- 复制 Token
- 运行下面的单元格并粘贴 Token

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

如果您不想使用 Google Colab 或 Jupyter Notebook，则需要使用此命令：`huggingface-cli login`

然后，我们只需运行 `mlagents-push-to-hf`。

我们定义 4 个参数：

1. `--run-id`：训练运行 ID 的名称。
2. `--local-dir`：智能体保存的路径，格式为 results/<run_id 名称>，在我这里是 results/First Training。
3. `--repo-id`：您要创建或更新的 Hugging Face 仓库名称。格式为 <您的 Hugging Face 用户名>/<仓库名称>。
如果该仓库不存在，**它将自动创建**。
4. `--commit-message`：由于 HF 仓库是 Git 仓库，您需要定义一个提交信息。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/mlagentspushtohub.png" alt="Push to Hub"/>

例如：

`!mlagents-push-to-hf  --run-id="SnowballTarget1" --local-dir="./results/SnowballTarget1" --repo-id="ThomasSimonini/ppo-SnowballTarget"  --commit-message="First Push"`

In [ ]:
!mlagents-push-to-hf --run-id="SnowballTarget1" --local-dir="./results/SnowballTarget1" --repo-id="ThomasSimonini/ppo-SnowballTarget" --commit-message="First Push"

In [ ]:
!mlagents-push-to-hf  --run-id= # 添加您的运行 id  --local-dir= # 您的本地目录  --repo-id= # 您的仓库 id  --commit-message= # 您的提交信息

否则，如果一切顺利，您应该在过程结束时看到以下内容（但会是不同的 URL 😆）：

```
Your model is pushed to the hub. You can view your model here: https://huggingface.co/ThomasSimonini/ppo-SnowballTarget
```

这是指向您模型的链接，它包含说明如何使用它的模型卡、您的 Tensorboard 以及您的配置文件。**非常棒的一点是，它是一个 Git 仓库，这意味着您可以有不同的提交，通过新的推送来更新您的仓库等。**

但现在最精彩的部分来了：**能够在网上直观地看您的智能体玩游戏 👀。**

### 观看您的智能体玩游戏 👀

这一步很简单：

1. 转到这里：https://huggingface.co/spaces/ThomasSimonini/ML-Agents-SnowballTarget

2. 启动游戏并点击右下角的按钮将其设为全屏。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/snowballtarget_load.png" alt="Snowballtarget load"/>

1. 在步骤 1 中，输入您的用户名（您的用户名区分大小写：例如，我的用户名是 ThomasSimonini，而不是 thomassimonini 或 ThOmasImoNInI），然后点击搜索按钮。

2. 在步骤 2 中，选择您的模型仓库。

3. 在步骤 3 中，**选择您要回放哪个模型**：
  - 我有多个模型，因为我们每 500,000 个时间步保存一次模型。
  - 但因为我想要最新的，所以我选择 `SnowballTarget.onnx`。

👉 令人高兴的是，**您可以尝试使用不同步骤的模型来查看智能体的进步。**

不要犹豫，在 Discord 的 #rl-i-made-this 频道中分享您的智能体获得的最高分数吧 🔥

现在让我们尝试一个更难的环境，叫做 Pyramids (金字塔)...

## Pyramids 🏆 (金字塔)

### 下载环境压缩包并移动到 `./training-envs-executables/linux/` 目录下
- 我们的环境可执行文件打包在 zip 文件中。
- 我们需要下载它并将其放置在 `./training-envs-executables/linux/` 目录中。
- 我们使用 Linux 可执行文件是因为我们使用的是 Colab，而 Colab 机器的操作系统是 Ubuntu (Linux)。

我们使用 `wget` 从 https://huggingface.co/spaces/unity/ML-Agents-Pyramids/resolve/main/Pyramids.zip 下载 Pyramids.zip 文件。

In [ ]:
!wget "https://huggingface.co/spaces/unity/ML-Agents-Pyramids/resolve/main/Pyramids.zip" -O ./training-envs-executables/linux/Pyramids.zip

我们解压 executable.zip 文件。

In [ ]:
%%capture
!unzip -d ./training-envs-executables/linux/ ./training-envs-executables/linux/Pyramids.zip

确保您的文件是可访问的。

In [ ]:
!chmod -R 755 ./training-envs-executables/linux/Pyramids/Pyramids

### 修改 PyramidsRND 配置文件
- 与第一个自定义环境不同，**Pyramids 是由 Unity 团队制作的**。
- 所以 PyramidsRND 配置文件已经存在，并且位于 `./content/ml-agents/config/ppo/PyramidsRND.yaml`。
- 您可能会问为什么 PyramidsRND 中带有 "RND"。RND 代表 *Random Network Distillation*（随机网络蒸馏），这是一种产生好奇心奖励的方法。如果您想了解更多关于该技术的信息，我们写了一篇文章对其进行了解释：https://medium.com/data-from-the-trenches/curiosity-driven-learning-through-random-network-distillation-488ffd8e5938

对于这次训练，我们将修改一件事：
- 总训练步数超参数太高了，因为我们只需 100 万（1M）训练步数即可达到基准指标（平均奖励 = 1.75）。
👉 为此，我们打开 `config/ppo/PyramidsRND.yaml`，**并将 `max_steps` 修改为 `1000000`。**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/pyramids-config.png" alt="Pyramids config"/>

作为实验，您还应该尝试修改其他一些超参数，Unity 提供了非常[优秀的文档，在这里详细解释了每一个超参数](https://github.com/Unity-Technologies/ml-agents/blob/main/docs/Training-Configuration-File.md)。

我们现在准备开始训练我们的智能体 🔥。

### 训练智能体

训练将耗时 30 到 45 分钟，具体取决于您的机器性能，去喝杯 ☕️ 吧，这是你应得的 🤗。

In [ ]:
!mlagents-learn ./config/ppo/PyramidsRND.yaml --env=./training-envs-executables/linux/Pyramids/Pyramids --run-id="Pyramids Training" --no-graphics

### 将智能体推送到 🤗 Hub

- 现在我们已经训练好了智能体，我们**准备好将它推送到 Hub，以便能够在浏览器中可视化观看它玩游戏🔥。**

In [ ]:
!mlagents-push-to-hf  --run-id= # 添加您的运行 id  --local-dir= # 您的本地目录  --repo-id= # 您的仓库 id  --commit-message= # 您的提交信息

### 观看您的智能体玩游戏 👀

👉 https://huggingface.co/spaces/unity/ML-Agents-Pyramids

### 🎁 奖励部分：为什么不在其他环境中训练呢？
现在您已经知道如何使用 ML-Agents 训练智能体了，**为什么不尝试其他的环境呢？**

ML-Agents 提供了 17 种不同的环境，我们也在构建一些自定义环境。最好的学习方法就是亲自去尝试各种事物，玩得开心！

![cover](https://miro.medium.com/max/1400/0*xERdThTRRM2k_U9f.png)

您可以在这里找到 Unity 官方环境的完整列表 👉 https://github.com/Unity-Technologies/ml-agents/blob/develop/docs/Learning-Environment-Examples.md

关于可视化智能体的演示页面 👉 https://huggingface.co/unity

目前我们已经集成了：
- [Worm](https://huggingface.co/spaces/unity/ML-Agents-Worm) 演示：您在其中教**毛毛虫爬行**。
- [Walker](https://huggingface.co/spaces/unity/ML-Agents-Walker) 演示：您在其中教智能体**朝着目标行走**。

今天的内容就到这里。恭喜您完成本教程！

最好的学习方法是多加练习和尝试。为什么不尝试其他环境呢？ML-Agents 拥有 17 个不同的环境，甚至您还可以创建自己的环境？查看文档并尽情探索吧！

我们第 6 单元见 🔥,

## 保持学习，保持优秀 🤗